In [38]:
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
from matplotlib.patches import Rectangle
import psycopg2
from datetime import datetime
import talib as ta

In [49]:
from config import db_conn
conn = db_conn()

symbol = 'FEDERALBNK-I'
from_date = '2025-10-29'
to_date = '2025-10-30'

query = f"""
    SELECT date AT TIME ZONE 'Asia/Kolkata' AS local_time, *
    FROM idata_15min
    WHERE date(date) >= %s AND date(date) <= %s
    ORDER BY date ASC;
"""
params = (from_date, to_date)

cursor = conn.cursor()
cursor.execute(query, params)
rows = cursor.fetchall()
columns = [desc[0] for desc in cursor.description]

df = pd.DataFrame(rows, columns=columns)
df['date'] = df["local_time"]
df.sort_values(by="date", inplace=True)
# df["date"] = pd.to_datetime(df["date"])
df.drop(columns=["local_time", "id"], inplace=True)
numeric_cols = ["open", "high", "low", "close", "volume"]
df[numeric_cols] = df[numeric_cols].astype(float)



In [61]:
dates = pd.to_datetime(df['date']).dt.date.unique()
yesterdays_data = df[df['date'].dt.date == dates[0]]
todays_data = df[df['date'].dt.date == dates[1]]

yesterdays_data_volume = yesterdays_data.groupby('symbol')["volume"].sum().reset_index()
yesterdays_data_volume.rename(columns={"volume": "yesterday_volume"}, inplace=True)
yesterdays_data_volume

,symbol,yesterday_volume
0,360ONE-I,1977000.0
1,ABB-I,616125.0
2,ABCAPITAL-I,12282200.0
3,ADANIENSOL-I,7103700.0
4,ADANIENT-I,4687500.0
...,...,...
209,VEDL-I,27533300.0
210,VOLTAS-I,1239000.0
211,WIPRO-I,14085000.0
212,YESBANK-I,81108800.0


In [73]:
todays_data_volume = todays_data[todays_data["date"].dt.time <= datetime.strptime("09:15:00", "%H:%M:%S").time()]
todays_data_volume = todays_data_volume.groupby('symbol')["volume"].sum().reset_index()
todays_data_volume.rename(columns={"volume": "today_volume"}, inplace=True)
merged_volume = pd.merge(yesterdays_data_volume, todays_data_volume, on="symbol", how="inner")
merged_volume['volume_change_percent'] = 100-((merged_volume['yesterday_volume'] - merged_volume['today_volume']) / merged_volume['yesterday_volume']) * 100
merged_volume[merged_volume["volume_change_percent"] > 25].sort_values("volume_change_percent", ascending=False).reset_index()

,index,symbol,yesterday_volume,today_volume,volume_change_percent
0,163,POLICYBZR-I,2137800.0,1650600.0,77.210216
1,57,DRREDDY-I,4979375.0,3260625.0,65.482616
2,90,IIFL-I,4243800.0,2506350.0,59.059098
3,30,BHEL-I,37949625.0,19527375.0,51.456042
4,86,IDEA-I,795731175.0,405120300.0,50.911704
5,115,LICHSGFIN-I,4550000.0,1850000.0,40.659341
6,31,BIOCON-I,9767500.0,3850000.0,39.416432
7,95,INDUSTOWER-I,9866800.0,3743400.0,37.939352
8,149,OIL-I,2283400.0,826000.0,36.174126
9,118,LT-I,4659550.0,1634850.0,35.086006
